# Analysis on Spread of Asset Pairs

This notebook covers the analysis of the spread of asset pairs, and finding examples of best asset pairs

# Imports

In [2]:
import sys
import os
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore # spread analysis


# Getting Historical Prices

For cointegration-based spread analysis, we study stocks with strong economic 
relationships within the same sector. Same-sector pairs share common macroeconomic 
drivers — such as interest rates, commodity prices, and regulation — making them 
strong candidates for cointegration.

We focus on the following sectors and candidate pairs:

| Sector | Pair | Tickers |
|---|---|---|
| Technology (Semiconductors) | Nvidia / AMD | `NVDA.O` / `AMD.O` |
| Technology (Semiconductors) | Nvidia / TSMC | `NVDA.O` / `TSM.N` |
| Consumer Staples (Beverages) | Coca-Cola / PepsiCo | `KO.N` / `PEP.O` |
| Financials (Banking) | JPMorgan / Bank of America | `JPM.N` / `BAC.N` |
| Financials (Banking) | Goldman Sachs / Morgan Stanley | `GS.N` / `MS.N` |
| Energy | ExxonMobil / Chevron | `XOM.N` / `CVX.N` |
| E-Commerce / Cloud | Amazon / Microsoft | `AMZN.O` / `MSFT.O` |
| E-Commerce / Cloud | Meta / Alphabet | `META.O` / `GOOGL.O` |
| Healthcare Pharma | Johnson & Johnson / Pfizer | `JNJ.O` / `PFE.O` |


We retrieve daily closing prices for all tickers over the full date range, 
then split into in-sample (training) and out-of-sample (test) sets for 
validation:

- **Training period**: 1 January 2020 – 31 December 2023
- **Test period**: 1 January 2024 – 31 December 2025


## Get Market Cap
Obtain market cap prices as of 29th December 2023 (last trading day of 2023 - end of test period)

In [3]:
market_cap_df = get_market_cap(
    TICKERS,
    start="2023-12-29",
    end="2023-12-29",
)

market_cap_df

[cache hit] AMD-O_AMZN-O_BAC-N_CVX-N_GOOGL-O_GS-N_JN__1D__4c761d2e99.csv


,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2023-12-29,1223193400000,2.381407e+11,5.011967e+11,2.547788e+11,2.335069e+11,4.917605e+11,2.664554e+11,1.258044e+11,153052304835,3.995975e+11,2.807263e+11,1.570153e+12,2.794828e+12,9.096286e+11,1755459040000,3.773169e+11,1.625602e+11


## Obtain ticker prices

Obtain ticker prices for all TICKER symbols from start of training period (2019-01-01) to end of training period (2023-12-31) date with interval of 1 day

In [4]:
prices_df = get_close_prices(
    TICKERS,
    start=TRAIN_START,
    end=TRAIN_END,
    interval=INTERVAL
)

[cache partial] Fetching 2019-01-01 → 2019-01-01
LSEG session opened.
[cache partial] Skipping (no trading data): LSEG returned no data for ['NVDA.O', 'AMD.O', 'TSM.N', 'KO.N', 'PEP.O', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N'] (2019-01-01 to 2019-01-01)
[cache partial] Fetching 2023-12-30 → 2023-12-31
[cache partial] Skipping (no trading data): LSEG returned no data for ['NVDA.O', 'AMD.O', 'TSM.N', 'KO.N', 'PEP.O', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N'] (2023-12-30 to 2023-12-31)


In [5]:
print(prices_df.shape)
prices_df.head()

(1258, 17)


,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2019-01-02,3.40550,18.83,36.52,46.93,109.28,99.31,24.96,172.03,40.40,69.69,110.69,76.9565,101.12,135.68,52.7340,127.75,40.998794
2019-01-03,3.19975,17.05,34.36,46.64,108.26,97.11,24.56,169.51,39.68,68.62,108.57,75.0140,97.40,131.74,51.2735,125.72,39.851776
2019-01-04,3.40475,19.00,34.97,47.57,110.48,100.69,25.58,175.05,41.30,71.15,110.82,78.7695,101.93,137.95,53.9035,127.83,40.761807
2019-01-07,3.58500,20.57,35.23,46.95,109.53,100.76,25.56,176.02,41.71,71.52,112.26,81.4755,102.06,138.05,53.7960,127.01,40.979835
2019-01-08,3.49575,20.75,34.94,47.48,110.58,100.57,25.51,175.37,41.45,72.04,111.77,82.8290,102.80,142.53,54.2685,129.96,41.169425


### Example

Plotting the closing price of NVDA.O from January 1, 2019 to December 31, 2023

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prices_df.index,
    y=prices_df['NVDA.O'],
    mode='lines',
    name='NVDA.O',
    line=dict(color='#76b900', width=1.5)
))

fig.update_layout(
    title='NVDA.O Close Price (Jan 2019 – Dec 2023)',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

# Spread Analysis

## Obtain Pre-Selected Candidate Pairs

In [7]:
CANDIDATE_PAIRS

[('NVDA.O', 'AMD.O'),
 ('NVDA.O', 'TSM.N'),
 ('KO.N', 'PEP.O'),
 ('JPM.N', 'BAC.N'),
 ('GS.N', 'MS.N'),
 ('XOM.N', 'CVX.N'),
 ('AMZN.O', 'MSFT.O'),
 ('META.O', 'GOOGL.O'),
 ('JNJ.N', 'PFE.N')]

## Run Engle-Granger Cointegration Tests on Candidate Pairs

Convert raw closing prices to log prices

In [8]:
prices_log = np.log(prices_df)

In [9]:
results_df = screen_pairs(prices_log, CANDIDATE_PAIRS, significance=0.05)

In [10]:
results_df

,y,x,hedge_ratio,intercept,adf_stat,p_value,is_cointegrated
0,CVX.N,XOM.N,0.713964,1.761593,-3.462651,0.009000,True
1,KO.N,PEP.O,0.664058,0.674101,-3.313506,0.014285,True
2,GS.N,MS.N,0.800362,2.256544,-3.252029,0.017164,True
3,AMD.O,NVDA.O,0.630470,2.596369,-2.769685,0.062736,False
4,JNJ.N,PFE.N,0.367091,3.691649,-2.675913,0.078300,False
5,AMZN.O,MSFT.O,0.500083,2.112186,-1.651058,0.456489,False
6,META.O,GOOGL.O,0.538279,2.974854,-1.508262,0.529421,False
7,TSM.N,NVDA.O,0.418804,3.261647,-1.433971,0.565886,False
8,BAC.N,JPM.N,0.879908,-0.798858,-0.982241,0.759686,False


### Filter Results

In [11]:
cointegrated_pairs = results_df[results_df['p_value'] < 0.05].copy()
print(cointegrated_pairs[['y', 'x', 'hedge_ratio', 'p_value']])


       y      x  hedge_ratio   p_value
0  CVX.N  XOM.N     0.713964  0.009000
1   KO.N  PEP.O     0.664058  0.014285
2   GS.N   MS.N     0.800362  0.017164


### Compute spreads

In [12]:
summaries = []
for _, row in cointegrated_pairs.iterrows():
    y = prices_log[row['y']]
    x = prices_log[row['x']]
    
    summary = spread_summary(
        y, x,
        hedge_ratio=row['hedge_ratio'],
        intercept=row['intercept'],
        window=60
    )
    summary['pair'] = f"{row['y']}/{row['x']}"
    summaries.append(summary)

summary_df = pd.DataFrame(summaries).set_index('pair')
summary_df


,half_life,hurst,current_zscore,spread_mean,spread_std,adf_stat,adf_pvalue,is_stationary
pair,,,,,,,,
CVX.N/XOM.N,33.850675,0.336406,0.628101,-4.021514e-15,0.062838,-3.462651,0.009000,True
KO.N/PEP.O,40.887317,0.418551,0.798284,-1.135710e-14,0.049604,-3.313506,0.014285,True
GS.N/MS.N,36.356403,0.397755,0.592609,5.946136e-15,0.051924,-3.252029,0.017164,True


## Visualise Spreads

### Visualise co-movements

In [30]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']} vs {row['x']} — Normalised Log Prices"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    y = prices_log[row['y']]
    x = prices_log[row['x']]
    
    # Normalise to 0 at start
    y_norm = y - y.iloc[0]
    x_norm = x - x.iloc[0]

    fig.add_trace(go.Scatter(
        x=y_norm.index, y=y_norm.values,
        name=row['y'], line=dict(width=1.5)
    ), row=i, col=1)
    
    fig.add_trace(go.Scatter(
        x=x_norm.index, y=x_norm.values,
        name=row['x'], line=dict(width=1.5, dash='dash')
    ), row=i, col=1)

fig.update_layout(
    height=900,
    title_text="Normalised Log Price Series — Cointegrated Pairs (Jan 2019 – Dec 2023)",
    showlegend=True
)
fig.show()

#### Load to matplotlib

In [1]:
import matplotlib.dates as mdates

plt.style.use("seaborn-v0_8-whitegrid")

fig, axes = plt.subplots(
    nrows=len(cointegrated_pairs),
    ncols=1,
    figsize=(10, 9),
    sharex=True
)

if len(cointegrated_pairs) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, cointegrated_pairs.iterrows()):
    y = prices_log[row["y"]]
    x = prices_log[row["x"]]

    y_norm = y - y.iloc[0]
    x_norm = x - x.iloc[0]

    ax.plot(y_norm.index, y_norm.values, label=row["y"], linewidth=1.5)
    ax.plot(x_norm.index, x_norm.values, label=row["x"], linestyle="--", linewidth=1.5)

    ax.set_title(f"{row['y']} vs {row['x']} — Normalised Log Prices", fontsize=11)
    ax.set_ylabel("Log price (normalised)")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].legend(loc="upper left", ncol=2, fontsize=9)

fig.suptitle(
    "Normalised Log Price Series — Cointegrated Pairs (Jan 2019 – Dec 2023)",
    fontsize=14
)
fig.tight_layout()
fig.savefig("figures/normalised_prices.pdf", bbox_inches="tight")
plt.close(fig)

NameError: name 'plt' is not defined

### Spread Characterisation

In [13]:
from plotly.subplots import make_subplots

pairs      = summary_df.index.tolist()
half_lives = summary_df['half_life'].tolist()
hursts     = summary_df['hurst'].tolist()
zscores    = summary_df['current_zscore'].tolist()
adf_pvals  = summary_df['adf_pvalue'].tolist()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Half-Life of Mean Reversion (Trading Days)",
        "Hurst Exponent",
        "Current Z-Score (End of Training Window)",
        "ADF p-value on Spread"
    )
)

# --- Half-life bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=half_lives,
    marker_color=['green' if h < 60 else 'orange' for h in half_lives],
    text=[f"{h:.1f}d" for h in half_lives], textposition='auto',
    name='Half-Life'
), row=1, col=1)
fig.add_hline(y=60, line=dict(color='red', dash='dash'), annotation_text='60d threshold', row=1, col=1)

# --- Hurst exponent bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=hursts,
    marker_color=['green' if h < 0.5 else 'red' for h in hursts],
    text=[f"{h:.3f}" for h in hursts], textposition='auto',
    name='Hurst'
), row=1, col=2)
fig.add_hline(y=0.5, line=dict(color='red',    dash='dash'), annotation_text='Random walk (0.5)', row=1, col=2)

# --- Current z-score bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=zscores,
    marker_color=['red' if abs(z) > 2 else 'steelblue' for z in zscores],
    text=[f"{z:.2f}" for z in zscores], textposition='auto',
    name='Z-Score'
), row=2, col=1)
fig.add_hline(y=2,  line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=-2, line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=0,  line=dict(color='gray',   dash='dash'), row=2, col=1)

# --- ADF p-value bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=adf_pvals,
    marker_color=['green' if p < 0.05 else 'red' for p in adf_pvals],
    text=[f"{p:.3f}" for p in adf_pvals], textposition='auto',
    name='ADF p-value'
), row=2, col=2)
fig.add_hline(y=0.05, line=dict(color='red', dash='dash'), annotation_text='5% significance', row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Spread Characterisation Summary — Cointegrated Pairs",
    showlegend=False
)

fig.show()

#### add to markdown

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

pairs      = summary_df.index.tolist()
half_lives = summary_df["half_life"].tolist()
hursts     = summary_df["hurst"].tolist()
zscores    = summary_df["current_zscore"].tolist()
adf_pvals  = summary_df["adf_pvalue"].tolist()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
ax_hl, ax_hu, ax_z, ax_adf = axes.flatten()

# Half-life
colors_hl = ["green" if h < 60 else "orange" for h in half_lives]
ax_hl.bar(pairs, half_lives, color=colors_hl)
for x, y in zip(pairs, half_lives):
    ax_hl.text(x, y, f"{y:.1f}d", ha="center", va="bottom", fontsize=9)
ax_hl.axhline(60, color="red", linestyle="--", linewidth=1)
ax_hl.text(len(pairs)-0.1, 60, "60d threshold", ha="right", va="bottom", color="red")
ax_hl.set_title("Half-Life of Mean Reversion (Trading Days)")
ax_hl.set_ylabel("Days")

# Hurst
colors_hu = ["green" if h < 0.5 else "red" for h in hursts]
ax_hu.bar(pairs, hursts, color=colors_hu)
for x, y in zip(pairs, hursts):
    ax_hu.text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=9)
ax_hu.axhline(0.5, color="red", linestyle="--", linewidth=1)
ax_hu.text(len(pairs)-0.1, 0.5, "Random walk (0.5)", ha="right", va="bottom", color="red")
ax_hu.set_title("Hurst Exponent")
ax_hu.set_ylabel("H")

# Current Z-score
colors_z = ["red" if abs(z) > 2 else "steelblue" for z in zscores]
ax_z.bar(pairs, zscores, color=colors_z)
for x, y in zip(pairs, zscores):
    ax_z.text(x, y, f"{y:.2f}", ha="center", va=("bottom" if y >= 0 else "top"), fontsize=9)
ax_z.axhline(0, color="gray", linestyle="--", linewidth=1)
ax_z.axhline(2, color="orange", linestyle=":", linewidth=1)
ax_z.axhline(-2, color="orange", linestyle=":", linewidth=1)
ax_z.set_title("Current Z-Score (End of Training Window)")
ax_z.set_ylabel("Z-score")

# ADF p-value
colors_adf = ["green" if p < 0.05 else "red" for p in adf_pvals]
ax_adf.bar(pairs, adf_pvals, color=colors_adf)
for x, y in zip(pairs, adf_pvals):
    ax_adf.text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=9)
ax_adf.axhline(0.05, color="red", linestyle="--", linewidth=1)
ax_adf.text(len(pairs)-0.1, 0.05, "5% significance", ha="right", va="bottom", color="red")
ax_adf.set_title("ADF p-value on Spread")
ax_adf.set_ylabel("p-value")

for ax in axes.flatten():
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Spread Characterisation Summary — Cointegrated Pairs", fontsize=14)
fig.tight_layout()
fig.savefig("figures/spread_characterisation.pdf", bbox_inches="tight")
plt.close(fig)

### Spread level between +1 std to +2 std

In [19]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Spread Level" 
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']], 
                           row['hedge_ratio'], row['intercept'])
    
    # Spread line
    fig.add_trace(go.Scatter(
        x=spread.index, y=spread.values, 
        line=dict(width=1, color='steelblue'), 
        name='Spread'), row=i, col=1)

    # Mean
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean()] * len(spread),
        line=dict(color='red', dash='dash', width=1),
        name='Mean', showlegend=(i == 1)), row=i, col=1)

    # ±1σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='+1σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='-1σ', showlegend=(i == 1)), row=i, col=1)

    # ±2σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='+2σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='-2σ', showlegend=(i == 1)), row=i, col=1)

fig.update_layout(
    height=900, 
    title_text="Spread Levels with σ Bands", 
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

#### Save as matplotlib

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(10, 9), sharex=True)

if n_pairs == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, cointegrated_pairs.iterrows()):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()

    mu = spread.mean()
    sigma = spread.std()

    ax.plot(spread.index, spread.values, color="steelblue", linewidth=1, label="Spread")
    ax.axhline(mu, color="red", linestyle="--", linewidth=1, label="Mean")

    ax.axhline(mu + sigma,  color="orange", linestyle=":", linewidth=1, label="+1σ")
    ax.axhline(mu - sigma,  color="orange", linestyle=":", linewidth=1, label="-1σ")
    ax.axhline(mu + 2*sigma, color="red", linestyle=":", linewidth=1, label="+2σ")
    ax.axhline(mu - 2*sigma, color="red", linestyle=":", linewidth=1, label="-2σ")

    ax.set_title(f"{row['y']}/{row['x']} — Spread Level")
    ax.set_ylabel("Spread")

axes[-1].set_xlabel("Date")
axes[0].legend(loc="upper left", ncol=3, fontsize=8)
fig.suptitle("Spread Levels with σ Bands", fontsize=14)
fig.tight_layout()
fig.savefig("figures/spread_levels.pdf", bbox_inches="tight")
plt.close(fig)

### Z-scores visualisation

In [21]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Rolling Z-Score (60d)"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept'])
    zscore = compute_zscore(spread, window=60)

    fig.add_trace(go.Scatter(x=zscore.index, y=zscore.values,
                             line=dict(width=1, color='purple')), row=i, col=1)
    fig.add_hline(y=0,  line=dict(color='red',    dash='dash'), row=i, col=1)
    fig.add_hline(y=2,  line=dict(color='orange', dash='dot'),  row=i, col=1)
    fig.add_hline(y=-2, line=dict(color='orange', dash='dot'),  row=i, col=1)

fig.update_layout(height=900, title_text="Rolling Z-Scores", showlegend=False)
fig.show()


#### Save to Matplotlib

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

window = 60
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(10, 9), sharex=True)

if n_pairs == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, cointegrated_pairs.iterrows()):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()

    zscore = compute_zscore(spread, window=window).dropna()

    ax.plot(zscore.index, zscore.values, color="purple", linewidth=1)
    ax.axhline(0,  color="red",    linestyle="--", linewidth=1)
    ax.axhline(2,  color="orange", linestyle=":",  linewidth=1)
    ax.axhline(-2, color="orange", linestyle=":",  linewidth=1)

    ax.set_title(f"{row['y']}/{row['x']} — Rolling Z-Score ({window}d)")
    ax.set_ylabel("Z-score")

axes[-1].set_xlabel("Date")
fig.suptitle("Rolling Z-Scores", fontsize=14)
fig.tight_layout()
fig.savefig("figures/spread_zscores.pdf", bbox_inches="tight")
plt.close(fig)

### Spread Distribution

In [27]:
from scipy import stats

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f"{row['y']}/{row['x']}" for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept']).dropna()

    # Histogram
    fig.add_trace(go.Histogram(x=spread.values, nbinsx=50, 
                               histnorm='probability density',
                               marker_color='steelblue', opacity=0.7,
                               name='Spread'), row=1, col=i)

    # Normal overlay
    x_range = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())
    fig.add_trace(go.Scatter(x=x_range, y=normal_pdf,
                             line=dict(color='red', width=2),
                             name='Normal fit'), row=1, col=i)

fig.update_layout(height=400, title_text="Spread Distributions", showlegend=False)
fig.show()


#### Save to Matplotlib

In [ ]:
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")

n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(
    nrows=1,
    ncols=n_pairs,
    figsize=(12, 4),
    sharey=True
)

if n_pairs == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, cointegrated_pairs.iterrows()):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()

    # Histogram (PDF)
    ax.hist(
        spread.values,
        bins=50,
        density=True,
        alpha=0.7,
        color="steelblue",
        label="Spread"
    )

    # Normal overlay
    x_range = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())
    ax.plot(x_range, normal_pdf, color="red", linewidth=2, label="Normal fit")

    ax.set_title(f"{row['y']}/{row['x']}")
    ax.set_xlabel("Spread")

axes[0].set_ylabel("Density")
axes[0].legend(fontsize=8)
fig.suptitle("Spread Distributions", fontsize=14)
fig.tight_layout()
fig.savefig("figures/spread_distributions.pdf", bbox_inches="tight")
plt.close(fig)